# TAD case studies and confounder checks

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import glob
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import scipy.stats
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig4_S19_S20_S22_S23_neurips2021_cite_bmmc_fc_interaction_gene_layer_tad_analysis")
os.makedirs(_OUTDIR_ABS, exist_ok=True)
PRIMARY = "../.experiment_data/primary"
CHIP = "../.experiment_data/reruns/fig4_tad/chip"
RESOURCES = "../preprocessing/resources"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

## Loading

In [ ]:
SEEDS = [11, 13, 14]
AUROC, HELDOUT = "predictors_label_valid_roc", "predictors_label_test_roc"
FOLDER = {"go": "calc_go_terms_cell_type", "genes": "calc_genes_cell_type", "groups": "calc_gene_groups_cell_type"}

networkvi_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}

networkvi_no_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}

networkvi_alternative_tad_map_paths = {
    "ENCFF804SET": {
        11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF804SET_gaf_standard_with_rna_s11/default",
        13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF804SET_gaf_standard_with_rna_s13/default",
        14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF804SET_gaf_standard_with_rna_s14/default",
    },
    "ENCFF164AGX": {
        11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF164AGX_gaf_standard_with_rna_s11/default",
        13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF164AGX_gaf_standard_with_rna_s13/default",
        14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF164AGX_gaf_standard_with_rna_s14/default",
    },
    "ENCFF203AKP": {
        11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF203AKP_gaf_standard_with_rna_s11/default",
        13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF203AKP_gaf_standard_with_rna_s13/default",
        14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF203AKP_gaf_standard_with_rna_s14/default",
    },
    "ENCFF173VDJ_shifted_benchmark": {
        11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_shifted_benchmark_gaf_standard_with_rna_s11/default",
        13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_shifted_benchmark_gaf_standard_with_rna_s13/default",
        14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_shifted_benchmark_gaf_standard_with_rna_s14/default",
    },
}


def run_dir(arch, seed, tad_map="ENCFF173VDJ"):
    if arch == "notad":
        return networkvi_no_tad_paths[seed]
    if tad_map == "ENCFF173VDJ":
        return networkvi_tad_paths[seed]
    return networkvi_alternative_tad_map_paths[tad_map][seed]


def cell_type_key(label):
    return label.replace("/", "_").replace(" ", "_")


def nice(ct):
    return ct.replace("_", " ").replace("G M prog", "G/M prog").replace("MK E prog", "MK/E prog")


def load_importances(directory, kind, modality="expression"):
    out = {}
    for f in sorted(glob.glob(f"{directory}/{FOLDER[kind]}/{modality}_cell_type_vs_rest_*_evaluated_graph_csv_with_ci.pkl")):
        label = os.path.basename(f).split("_vs_rest_", 1)[1].split("_res_unp_act_mask")[0]
        out[cell_type_key(label)] = pd.read_pickle(f).apply(pd.to_numeric, errors="coerce")
    return out


def paired(tad, base, column=AUROC):
    return {ct: pd.concat([tad[ct][column].rename("tad"), base[ct][column].rename("base")], axis=1, join="inner") for ct in sorted(set(tad) & set(base))}


def pct(x):
    return 100 * (np.asarray(x) - 1)


def boot_ci(x):
    r = scipy.stats.bootstrap((np.asarray(x, dtype=float),), np.mean, n_resamples=10000, confidence_level=0.95, random_state=0)
    return r.confidence_interval.low, r.confidence_interval.high


def averaged(tables):
    cts = sorted(set.intersection(*[set(t) for t in tables]))
    return {ct: pd.concat([t[ct][AUROC] for t in tables], axis=1, join="inner").mean(axis=1).rename(AUROC).to_frame() for ct in cts}


def gain_ci(ratios):
    lo, hi = boot_ci(ratios)
    return {"value (%)": float(pct(np.mean(ratios))), "95% CI low (%)": float(pct(lo)), "95% CI high (%)": float(pct(hi)), "Wilcoxon p": wilcoxon(ratios, np.ones_like(ratios)).pvalue}

In [ ]:
IMP = {(kind, arch, s): load_importances(run_dir(arch, s), kind) for kind in FOLDER for arch in ["tad", "notad"] for s in SEEDS}
PAIRS = {(kind, s): paired(IMP[(kind, "tad", s)], IMP[(kind, "notad", s)]) for kind in FOLDER for s in SEEDS}
AVG = {(kind, arch): averaged([IMP[(kind, arch, s)] for s in SEEDS]) for kind in FOLDER for arch in ["tad", "notad"]}
PA = {kind: paired(AVG[(kind, "tad")], AVG[(kind, "notad")]) for kind in FOLDER}
HELD = {s: paired(IMP[("groups", "tad", s)], IMP[("groups", "notad", s)], HELDOUT) for s in SEEDS}
pd.DataFrame({f"{kind} {arch} s{s}": [len(v), len(next(iter(v.values())))] for (kind, arch, s), v in IMP.items()}, index=["cell types", "nodes"]).T

## Expression

In [ ]:
import muon as mu
import scipy.sparse as sp

rna = mu.read_h5ad(f"{PRIMARY}/preprocessing/datasets/neurips2021_cite_BMMC/data/neurips2021_cite_BMMC.h5mu", "rna")
counts = rna.layers["counts"]
counts = counts.tocsr() if sp.issparse(counts) else sp.csr_matrix(counts)
library_size = np.asarray(counts.sum(axis=1)).ravel()
hvg = rna.var["highly_variable"].values
counts_hvg = counts[:, hvg]
lognorm = sp.diags(1e4 / library_size) @ counts_hvg
lognorm.data = np.log1p(lognorm.data)
labels = rna.obs["cell_type"].astype(str).values
GENES = list(rna.var["gene_stable_id"].values[hvg])
frac, mean, n_cells = {}, {}, {}
for label in np.unique(labels):
    m = labels == label
    k = cell_type_key(label)
    frac[k] = np.asarray((counts_hvg[m] > 0).sum(axis=0)).ravel() / m.sum()
    mean[k] = np.asarray(lognorm[m].sum(axis=0)).ravel() / m.sum()
    n_cells[k] = int(m.sum())
FRAC, MEAN = pd.DataFrame(frac, index=GENES), pd.DataFrame(mean, index=GENES)
HVG_DISPERSION = rna.var.set_index("gene_stable_id")["dispersions_norm"].fillna(0).to_dict()
print(FRAC.shape)

## Annotation

In [ ]:
names = pd.read_csv(f"{RESOURCES}/ensembl_gene_id_gene_name_mapping/ensembl_gene_id_gene_name_mapping.csv")
gene_name = {g: n for g, n in zip(names["Gene stable ID"], names["Gene name"]) if isinstance(n, str) and n}
name = lambda g: gene_name.get(g, g)
tad_to_label = lambda t: ", ".join(name(g) for g in t.split("_"))
positions = pd.read_csv(f"{RESOURCES}/ensembl_gene_id_chromosome_position_mapping/ensembl_gene_id_chromosome_position_mapping.csv")


def tad_loci(tad_ids):
    pos = positions.groupby("Gene stable ID").agg({"Chromosome/scaffold name": "first", "Gene start (bp)": list, "Gene end (bp)": list})
    out = {}
    for t in tad_ids:
        genes = [g for g in t.split("_") if g in pos.index]
        if not genes:
            continue
        starts = [x for g in genes for x in pos.loc[g, "Gene start (bp)"]]
        ends = [x for g in genes for x in pos.loc[g, "Gene end (bp)"]]
        out[t] = ("chr" + str(pos.loc[genes[0], "Chromosome/scaffold name"]), min(starts), max(starts), max(ends))
    return out

In [ ]:
gene_id = {name(g): g for g in GENES}
panglao = pd.read_csv(f"{PRIMARY}/PanglaoDB_markers_27_Mar_2020.tsv", delimiter="\t")
MARKERS = set(names[names["Gene name"].isin(np.unique(panglao[panglao["organ"] == "Immune system"]["official gene symbol"]))]["Gene stable ID"])
go_annotation = pickle.load(open(f"{PRIMARY}/_biological_process_go-basic_goa_human_ensembl_gene_mapping_2024-08-08_goa_human_isoform_ensembl_gene_mapping_2024-08-08_goa_human_rna_ensembl_gene_mapping_2024-08-08.v2_processed.pkl", "rb"))
namespace, term = {}, None
for line in open(f"{PRIMARY}/preprocessing/resources/ebi_ensembl_go_mappings/go-basic.obo"):
    if line.startswith("[Term]"):
        term = None
    elif line.startswith("id: GO:"):
        term = line.split()[1]
    elif line.startswith("namespace: ") and term:
        namespace[term] = line.split()[1]
N_GO_BP = {g: len({t for t in go_annotation.get(g, set()) if namespace.get(t) == "biological_process"}) for g in GENES}
LINEAGE = {
    "lymphoid": ["Lymph_prog", "ILC", "ILC1", "NK", "NK_CD158e1+", "CD4+_T_naive", "CD4+_T_activated", "CD4+_T_activated_integrinB7+", "CD4+_T_CD314+_CD45RA+", "CD8+_T_naive",
                 "CD8+_T_naive_CD127+_CD26-_CD101-", "T_prog_cycling", "CD8+_T_CD49f+", "CD8+_T_CD69+_CD45RA+", "CD8+_T_CD69+_CD45RO+", "CD8+_T_TIGIT+_CD45RA+", "CD8+_T_TIGIT+_CD45RO+",
                 "CD8+_T_CD57+_CD45RA+", "CD8+_T_CD57+_CD45RO+", "gdT_CD158b+", "gdT_TCRVD2+", "dnT", "T_reg", "MAIT", "Naive_CD20+_B_IGKC+", "Naive_CD20+_B_IGKC-", "Transitional_B",
                 "B1_B_IGKC+", "B1_B_IGKC-", "Plasma_cell_IGKC+", "Plasma_cell_IGKC-", "Plasmablast_IGKC+", "Plasmablast_IGKC-"],
    "myeloid": ["G_M_prog", "cDC1", "cDC2", "pDC", "CD14+_Mono", "CD16+_Mono"],
    "erythroid/MK": ["MK_E_prog", "Erythroblast", "Normoblast", "Proerythroblast", "Reticulocyte"],
    "HSC": ["HSC"],
}
lineage_of = lambda ct: next((k for k, v in LINEAGE.items() if ct in v), "other")
print(len(MARKERS), "marker genes;", sum(v == 0 for v in N_GO_BP.values()), "of", len(N_GO_BP), "genes without GO biological-process annotation")

## GO importance

In [ ]:
FIG4B_TERMS = {"GO:0008150": "biological process", "GO:0002376": "immune system process", "GO:0050896": "response to stimulus", "GO:0002252": "immune effector process",
               "GO:0002520": "immune system development", "GO:0048534": "hematopoietic or lymphoid organ development", "GO:0002443": "leukocyte mediated immunity",
               "GO:0002444": "myeloid leukocyte mediated immunity", "GO:0002449": "lymphocyte mediated immunity", "GO:0002200": "somatic diversification of immune receptors",
               "GO:0043500": "muscle adaptation", "GO:0051775": "response to redox state", "GO:0014823": "response to activity", "GO:0009628": "response to abiotic stimulus"}
p_go = PA["go"]
per_term = np.array([np.mean(x.tad[x.base > 0] / x.base[x.base > 0]) for x in p_go.values()])
x16 = p_go["CD16+_Mono"]
GO_SUMMARY = pd.DataFrame({"GO importance gain, all cell types": gain_ci(per_term), "GO importance change, CD16+ Mono": gain_ci((x16.tad / x16.base).values)}).T
display(GO_SUMMARY)
print("CD16+ Mono: GO terms with positive change", int((x16.tad > x16.base).sum()), "of", len(x16), f"({100 * (x16.tad > x16.base).mean():.1f}%)")
rows = []
for t, label in FIG4B_TERMS.items():
    d = [PAIRS[("go", s)]["CD16+_Mono"].at[t, "tad"] - PAIRS[("go", s)]["CD16+_Mono"].at[t, "base"] for s in SEEDS]
    w = [IMP[("go", arch, s)]["CD16+_Mono"].at[t, "predictors_label_valid_roc_ci_upper"] - IMP[("go", arch, s)]["CD16+_Mono"].at[t, "predictors_label_valid_roc_ci_lower"] for arch in ["tad", "notad"] for s in SEEDS]
    r = dict(term=label, go=t, change=np.mean(d), ci_width_max=max(w))
    rows.append(r)
FIG4B = pd.DataFrame(rows)
print("max. 95% CI width:", round(FIG4B.ci_width_max.max(), 3))
FIG4B.round(3)

In [ ]:
per = [PAIRS[("go", s)]["CD16+_Mono"].pipe(lambda x: x.tad - x.base) for s in SEEDS]
idx = sorted(set.intersection(*[set(x.index) for x in per]))
go_diff_cd16_mono = np.mean([x.reindex(idx).values for x in per], axis=0)
set_figure_params("single", 2.2)
fig, ax = plt.subplots()
ax.hist(go_diff_cd16_mono, bins=40, color="#70acc0", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="#c12075", linewidth=0.8, linestyle="--")
n_pos = np.sum(go_diff_cd16_mono > 0)
n_neg = np.sum(go_diff_cd16_mono < 0)
ax.set_xlabel("GO importance change (model − baseline)")
ax.set_ylabel("Number of GO terms")
ax.set_title(f"CD16+ Monocytes: {n_pos} terms positive, {n_neg} terms negative")
plt.tight_layout()
for ext in ["pdf", "svg", "png"]:
    plt.savefig(os.path.join(_OUTDIR_ABS, f"go_diff_distribution_cd16.{ext}"), bbox_inches="tight", **({"dpi": 500} if ext == "png" else {}))
plt.show()

## Gene and TAD importance

In [ ]:
GENE_TAD_SUMMARY = pd.DataFrame({label: gain_ci(np.array([x.tad.mean() / x.base.mean() for x in PA[kind].values()])) for kind, label in [("genes", "gene importance gain"), ("groups", "TAD importance gain")]}).T
display(GENE_TAD_SUMMARY)
per_ct = []
for ct, x in PA["groups"].items():
    ratio = (x.tad / x.base).replace([np.inf, -np.inf], np.nan).dropna()
    per_ct.append(dict(cell_type=ct, lineage=lineage_of(ct), tad_change_pct=float(pct(ratio.mean()))))
TAD_PER_CT = pd.DataFrame(per_ct)
print("cell types with a positive TAD importance change:", int((TAD_PER_CT.tad_change_pct > 0).sum()), "of", len(TAD_PER_CT))
TAD_PER_CT.groupby("lineage").tad_change_pct.agg(["mean", "min", "size"])

## Marker genes

In [ ]:
gp, tp = PA["genes"], PA["groups"]
m = np.array([x.tad[x.index.isin(MARKERS)].mean() for x in gp.values()])
nm = np.array([x.tad[~x.index.isin(MARKERS)].mean() for x in gp.values()])
tm, tnm, gm, gnm = [], [], [], []
for x in tp.values():
    has = np.array([any(g in MARKERS for g in t.split("_")) for t in x.index])
    gain = x.tad - x.base
    tm.append(x.tad[has].mean())
    tnm.append(x.tad[~has].mean())
    gm.append(gain[has].mean())
    gnm.append(gain[~has].mean())
MARKER_SUMMARY = pd.DataFrame({"marker vs other genes": gain_ci(m / nm), "TADs with vs without markers": gain_ci(np.array(tm) / np.array(tnm)), "gain, TADs with vs without markers": gain_ci(np.array(gm) / np.array(gnm))}).T
MARKER_SUMMARY

## Alternative TAD maps

In [ ]:
SOURCES = {"ENCFF804SET": "T47D", "ENCFF164AGX": "MCF7", "ENCFF203AKP": "GM12878", "ENCFF173VDJ_shifted_benchmark": "position-shifted K562"}
SEVEN = ["CD14+_Mono", "CD16+_Mono", "Naive_CD20+_B_IGKC+", "NK", "pDC", "CD4+_T_naive", "CD8+_T_naive"]
rows = {}
for src, label in SOURCES.items():
    for kind, kl in [("genes", "gene importance"), ("groups", "TAD-group importance")]:
        p = {ct: x for ct, x in paired(averaged([load_importances(run_dir("tad", s, src), kind) for s in SEEDS]), AVG[(kind, "tad")]).items() if ct in SEVEN}
        assert len(p) == len(SEVEN)
        r = gain_ci(np.array([x.tad.mean() / x.base.mean() for x in p.values()]))
        r.pop("Wilcoxon p")
        rows[f"{label}, {kl}"] = r
ALT_SUMMARY = pd.DataFrame(rows).T
ALT_SUMMARY

## TAD importance gain

In [ ]:
DIFFS = {}
for s in SEEDS:
    p = PAIRS[("groups", s)]
    common = sorted(set.intersection(*[set(x.index) for x in p.values()]))
    DIFFS[s] = pd.DataFrame({ct: (x.tad - x.base).reindex(common) for ct, x in p.items()})
common = sorted(set.intersection(*[set(d.index) for d in DIFFS.values()]))
cell_keys = sorted(set.intersection(*[set(d.columns) for d in DIFFS.values()]), key=nice)
D = sum(d.reindex(index=common, columns=cell_keys) for d in DIFFS.values()) / len(SEEDS)
tad_ids_ordered = list(D.index)
phenotypes_labels = [nice(c) for c in cell_keys]
mean_gain_per_tad = D.mean(axis=1).values
print(D.shape)

In [ ]:
def find_tad(gene_names):
    return next(t for t in tad_ids_ordered if {name(g) for g in t.split("_")} == set(gene_names))


def tad_table(t):
    rows = []
    for ct in cell_keys:
        a = np.array([PAIRS[("groups", s)][ct].at[t, "tad"] for s in SEEDS])
        b = np.array([PAIRS[("groups", s)][ct].at[t, "base"] for s in SEEDS])
        h = np.array([HELD[s][ct].at[t, "tad"] - HELD[s][ct].at[t, "base"] for s in SEEDS])
        g = a - b
        r = dict(cell_type=ct, n_cells=n_cells.get(ct, 0), before=b.mean(), after=a.mean(), gain=g.mean(), gain_sd=g.std(ddof=1))
        r.update({f"gain_s{s}": x for s, x in zip(SEEDS, g)})
        r.update({f"heldout_gain_s{s}": x for s, x in zip(SEEDS, h)})
        r["gain_rank_pct"] = 100 * float((D[ct] > D.at[t, ct]).mean())
        r["n_expressed"] = int(sum(FRAC.at[x, ct] >= 0.2 for x in t.split("_") if x in FRAC.index))
        r["member_detection"] = "; ".join(f"{name(x)} {100 * FRAC.at[x, ct]:.0f}%" for x in t.split("_") if x in FRAC.index)
        rows.append(r)
    out = pd.DataFrame(rows).sort_values("after", ascending=False).reset_index(drop=True)
    out.insert(2, "importance_rank", np.arange(1, len(out) + 1))
    return out


T1 = find_tad(["HLA-DMA", "HLA-DMB"])
T2 = find_tad(["YBX3", "KLRD1", "KLRC1", "GABARAPL1", "KLRC4", "KLRC2", "KLRC3", "KLRK1", "KLRK1-AS1", "LINC02446"])
KLR_CORE = ["KLRD1", "KLRK1", "KLRK1-AS1", "KLRC4", "KLRC3", "KLRC2", "KLRC1", "LINC02446"]
ADJ = ["GABARAPL1", "YBX3"]
print("TAD 1:", T1, "| TAD 2:", T2)

## Gene–gene interaction weights

In [ ]:
import torch

acc = []
for s in SEEDS:
    d = torch.load(f"{run_dir('tad', s)}/model.pt", map_location="cpu", weights_only=False)
    nk = d["attr_dict"]["init_params_"]["non_kwargs"]
    B, ids = nk["standard_gene_size"], list(nk["ensembl_ids_genes"])
    k = "z_encoder_expression.genemodel.genemodel.interaction_layer.0.block_sparse"
    idx, val = d["model_state_dict"][k + ".indices"].numpy(), np.abs(d["model_state_dict"][k + ".values"].numpy())
    df = pd.DataFrame({"r": idx[0] // B, "c": idx[1] // B, "w": val})
    df = df[df.r != df.c]
    df["pair"] = [tuple(sorted((ids[a], ids[b]))) for a, b in zip(df.r, df.c)]
    acc.append(df.groupby("pair").w.mean().rename(s))
W = pd.concat(acc, axis=1).mean(axis=1).sort_values(ascending=False)
PAIR_WEIGHTS = pd.DataFrame({"pair": [f"{name(a)}–{name(b)}" for a, b in W.index], "mean_weight": W.values, "rank": W.rank(ascending=False, method="min").astype(int).values})
print(len(PAIR_WEIGHTS), "gene pairs")
display(PAIR_WEIGHTS.head(10))
display(PAIR_WEIGHTS[PAIR_WEIGHTS.pair.isin(["CD4–LAG3", "LAG3–CD4", "H3C6–H2BC9", "H2BC9–H3C6", "CCR10–ENSG00000267042", "ENSG00000267042–CCR10"])])

## Example TADs

In [ ]:
t1 = tad_table(T1)
display(t1.round(3))
t2 = tad_table(T2)
display(t2.round(3))

In [ ]:
rows = []
for g in ["LINC02446", "KLRK1-AS1", "KLRD1", "KLRK1"]:
    for ct in ["NK", "gdT_CD158b+", "CD8+_T_TIGIT+_CD45RO+", "CD8+_T_CD69+_CD45RA+", "Reticulocyte", "CD14+_Mono"]:
        a = [IMP[("genes", "tad", s)][ct].at[gene_id[g], AUROC] for s in SEEDS]
        b = [IMP[("genes", "notad", s)][ct].at[gene_id[g], AUROC] for s in SEEDS]
        rows.append(dict(gene=g, go_bp_terms=N_GO_BP[gene_id[g]], cell_type=ct, detection=FRAC.at[gene_id[g], ct], before=np.mean(b), after=np.mean(a), gain_per_seed=" / ".join(f"{x - y:+.3f}" for x, y in zip(a, b))))
pd.DataFrame(rows).round(3)

In [ ]:
MAPS = {"K562": "ENCFF173VDJ_K562", "GM12878": "ENCFF203AKP_GM12878", "MCF-7": "ENCFF164AGX_MCF-7"}
genes_all = ADJ[:1] + KLR_CORE + ADJ[1:]
membership = pd.DataFrame("", index=genes_all, columns=list(MAPS))
for label, stem in MAPS.items():
    groups = np.load(glob.glob(f"{PRIMARY}/data/tads/encode/{stem}_GRCh38_contact_domains_juicertools_gene_groups.npy")[0], allow_pickle=True)
    hits = [sorted({name(x) for x in gr} & set(genes_all)) for gr in groups if {name(x) for x in gr} & set(genes_all)]
    big = max(hits, key=len) if hits else []
    for g in genes_all:
        membership.loc[g, label] = "KLR domain" if g in big else ("separate domain" if any(g in h for h in hits) else "no domain")
membership

## Gene drop-out

In [ ]:
DROP_CASES = [(T1, "Naive_CD20+_B_IGKC+"), (T2, "CD8+_T_TIGIT+_CD45RO+"), (T2, "CD8+_T_CD69+_CD45RA+")]
DROP = {}
for arch in ["tad", "notad"]:
    for s in SEEDS:
        d = f"{run_dir(arch, s)}/calc_gene_groups_cell_type"
        files = sorted(glob.glob(f"{d}/perturbation_ENSG*_zero_named_jeffreys_ci.pkl")) + sorted(glob.glob(f"{d}/perturbation_ENSG*_zero_named_jeffreys_ci_sub*.pkl"))
        vals, last_index, rows_of = {}, None, None
        for f in files:
            gene = os.path.basename(f).split("_")[1]
            for label, x in pickle.load(open(f, "rb"))["expression"].items():
                ct = cell_type_key(label)
                if not any(ct == c for _, c in DROP_CASES):
                    continue
                if last_index is None or not x.index.equals(last_index):
                    keys = [frozenset(i.split("_")) for i in x.index]
                    last_index, rows_of = x.index, {t: [j for j, k in enumerate(keys) if k == frozenset(t.split("_"))] for t in [T1, T2]}
                col = x[AUROC].astype(float)
                for t, c in DROP_CASES:
                    if c == ct and rows_of[t]:
                        vals[(gene, ct, t)] = float(col.iloc[rows_of[t]].mean())
        DROP[(arch, s)] = vals


def dropout_table(t, ct):
    full = np.array([PAIRS[("groups", s)][ct].at[t, "tad"] - PAIRS[("groups", s)][ct].at[t, "base"] for s in SEEDS])
    genes = set.intersection(*[{g for g, c, x in DROP[(a, s)] if c == ct and x == t} for a in ["tad", "notad"] for s in SEEDS])
    rows = []
    for g in genes:
        gd = np.array([DROP[("tad", s)][(g, ct, t)] - DROP[("notad", s)][(g, ct, t)] for s in SEEDS])
        td = np.mean([PAIRS[("groups", s)][ct].at[t, "tad"] - DROP[("tad", s)][(g, ct, t)] for s in SEEDS])
        nd = np.mean([PAIRS[("groups", s)][ct].at[t, "base"] - DROP[("notad", s)][(g, ct, t)] for s in SEEDS])
        r = dict(dropped_gene=name(g), relation="member" if g in t.split("_") else "outside partner", detection=FRAC.at[g, ct] if g in FRAC.index else np.nan,
                 share=1 - gd.mean() / full.mean(), delta_tad_auroc=td, delta_notad_auroc=nd)
        r.update({f"share_s{s}": (f_ - x) / f_ for s, f_, x in zip(SEEDS, full, gd)})
        rows.append(r)
    return pd.DataFrame(rows).sort_values("share", ascending=False).reset_index(drop=True)


for t, ct in DROP_CASES:
    print(tad_to_label(t), "|", ct)
    display(dropout_table(t, ct).round(3))

## Figure 4d and 4e

In [ ]:
NAVY, TEAL, PINK, GREY, RED, GREEN = "#003754", "#70acc0", "#c12075", "#9d9c9c", "#d0312d", "#4caf50"
CMAP_GAIN = LinearSegmentedColormap.from_list("gain", ["#ffffff", "#c9e3ea", TEAL, NAVY], N=256)
CMAP_EXPR = LinearSegmentedColormap.from_list("expr", ["#ffffff", "#f7c9b4", "#e8664a", "#a50f15"], N=256)
CMAP_DROP = LinearSegmentedColormap.from_list("drop", ["#ffffff", "#c9e3ea", TEAL, NAVY], N=256)
FIG4_RC = {"svg.fonttype": "none", "pdf.fonttype": 42, "font.size": 6, "axes.titlesize": 7, "axes.labelsize": 6, "xtick.labelsize": 6, "ytick.labelsize": 6,
           "legend.fontsize": 6, "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2, "ytick.major.size": 2, "axes.spines.top": False,
           "axes.spines.right": False, "svg.hashsalt": "networkvi"}
NICE4 = {"Naive_CD20+_B_IGKC+": "Naive CD20⁺ B IGKC⁺", "B1_B_IGKC+": "B1 B IGKC⁺", "cDC2": "cDC2", "pDC": "pDC", "CD14+_Mono": "CD14⁺ Mono", "CD16+_Mono": "CD16⁺ Mono",
         "CD8+_T_TIGIT+_CD45RO+": "CD8⁺ T TIGIT⁺ CD45RO⁺", "CD8+_T_CD69+_CD45RA+": "CD8⁺ T CD69⁺ CD45RA⁺", "gdT_CD158b+": "γδ T CD158b⁺"}
ROWS = list(NICE4)
E_GENES = ["HLA-DMA", "HLA-DMB", "KLRD1", "KLRK1", "KLRK1-AS1", "LINC02446"]
VMAX = {"TAD 1": 0.12, "TAD 2": 0.03}
CMAPS = {"TAD 1": CMAP_GAIN, "TAD 2": LinearSegmentedColormap.from_list("gain2", ["#ffffff", "#f3c6dc", "#d9609e", PINK], N=256)}
emax = 2.5
tt = {"TAD 1": t1.set_index("cell_type"), "TAD 2": t2.set_index("cell_type")}
G = pd.DataFrame({k: v.gain for k, v in tt.items()}).loc[ROWS]
SD = pd.DataFrame({k: v.gain_sd for k, v in tt.items()}).loc[ROWS]
MASK = pd.DataFrame({k: v.n_expressed for k, v in tt.items()}).loc[ROWS] < 2
CIW = {k: max(float(IMP[("groups", arch, s)][ct].at[t, "predictors_label_valid_roc_ci_upper"] - IMP[("groups", arch, s)][ct].at[t, "predictors_label_valid_roc_ci_lower"]) for arch in ["tad", "notad"] for s in SEEDS for ct in ROWS)
       for k, t in [("TAD 1", T1), ("TAD 2", T2)]}


def css_font(m):
    style, weight, size, family = m.group(1), m.group(2), m.group(3), m.group(4)
    out = f"font-family: {family}; font-size: {size}px"
    if style and style.strip() != "normal":
        out += f"; font-style: {style.strip()}"
    if weight:
        out += f"; font-weight: {weight.strip()}"
    return out


def save_fig4(fig, stem):
    for ext in ["svg", "png", "pdf"]:
        _orig_figure_savefig(fig, os.path.join(_OUTDIR_ABS, f"{stem}.{ext}"), dpi=400, bbox_inches=None, transparent=True)
    plt.close(fig)
    p = os.path.join(_OUTDIR_ABS, f"{stem}.svg")
    txt = re.sub(r"font: ((?:italic|oblique|normal) )?((?:bold|[1-9]00) )?([0-9.]+)px '([^']+)'", css_font, open(p).read())
    open(p, "w").write(txt)


def draw_fig4de():
    PT = 0.75
    FS, FS_CELL = 7.51 * PT, 6.1 * PT
    X0P, Y0P, WP, HP = 0, 345, 343, 330
    fx = lambda x: (x - X0P) / WP
    fy = lambda y: 1 - (y - Y0P) / HP
    box = lambda x0, y0, x1, y1: [fx(x0), fy(y1), fx(x1) - fx(x0), fy(y0) - fy(y1)]
    TOP, BOT, nR = 417, 592, len(ROWS)
    rowy = lambda i: TOP + (i + 0.5) * (BOT - TOP) / nR
    DX0, DX1, EX0, EX1 = 82, 138, 148, 335
    fig = plt.figure(figsize=(WP / 96, HP / 96))
    axd = fig.add_axes(box(DX0, TOP, DX1, BOT))
    axd.set_xlim(-0.5, 1.5)
    axd.set_ylim(nR - 0.5, -0.5)
    axd.axis("off")
    for j, k in enumerate(["TAD 1", "TAD 2"]):
        for i, ct in enumerate(ROWS):
            v = float(G.at[ct, k])
            lab = f"{round(v, 3) + 0.0:.3f}".replace("-", "−")
            if MASK.at[ct, k]:
                axd.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fc="#efefef", ec="none"))
                axd.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="////", ec="#c8c8c8", lw=0))
                axd.text(j, i, lab, ha="center", va="center", fontsize=FS_CELL, color="#9a9a9a", style="italic")
            else:
                x = min(max(v, 0.0) / VMAX[k], 1.0)
                axd.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fc=CMAPS[k](x), ec="none"))
                axd.text(j, i, lab, ha="center", va="center", fontsize=FS_CELL, color="white" if x > 0.6 else "#1b1f2a")
    for yb in np.arange(0.5, nR - 0.5, 1):
        axd.axhline(yb, color="grey", ls="--", lw=0.4)
    axd.axvline(0.5, color="grey", ls="--", lw=0.4)
    fig.text(fx(DX1), fy(351), "TAD importance gain\n(AUROC, after − before)", fontsize=FS, fontweight="bold", va="top", ha="right", linespacing=1.1)
    cw = (DX1 - DX0) / 2
    for j, k in enumerate(["TAD 1", "TAD 2"]):
        cx = DX0 + (j + 0.5) * cw
        fig.text(fx(cx), fy(372), k, fontsize=FS, va="top", ha="center")
        cax = fig.add_axes(box(cx - 4, 383, cx + 2, 411))
        cb = fig.colorbar(mpl.cm.ScalarMappable(norm=Normalize(0, VMAX[k]), cmap=CMAPS[k]), cax=cax, ticks=[0, VMAX[k]])
        cb.outline.set_visible(False)
        cax.tick_params(labelsize=FS, length=1.5, width=0.4, pad=1)
        cax.set_yticklabels(["0", f"{VMAX[k]:g}"], fontweight="bold")
    sd_int = {k: SD[k][~MASK[k]].max() for k in ["TAD 1", "TAD 2"]}
    LH = 7.2
    for j, (k, txt, y0) in enumerate([("TAD 1", "HLA-DMA, HLA-DMB;", 600), ("TAD 2", "KLR genes (GABARAPL1–YBX3);", 629)]):
        cx = DX0 + (j + 0.5) * cw
        fig.add_artist(mpl.lines.Line2D([fx(cx), fx(cx)], [fy(BOT), fy(y0 - 2)], color="black", lw=0.5))
        fig.text(fx(cx + 2), fy(y0), f"{k}:", fontsize=FS, va="top", ha="right", fontweight="bold")
        fig.text(fx(cx + 2), fy(y0 + LH), txt + "\nmax. 95% CI width: " + f"{CIW[k]:.2f}", fontsize=FS, va="top", ha="right", linespacing=1.05)
    for i, ct in enumerate(ROWS):
        fig.text(fx(DX0 - 3), fy(rowy(i)), NICE4[ct], fontsize=FS_CELL, va="center", ha="right")
    axe = fig.add_axes(box(EX0, TOP, EX1, BOT))
    nG = len(E_GENES)
    axe.set_xlim(-0.5, nG - 0.5)
    axe.set_ylim(nR - 0.5, -0.5)
    axe.set_xticks([])
    axe.set_yticks([])
    for sp_ in axe.spines.values():
        sp_.set_visible(True)
        sp_.set_linewidth(0.8)
        sp_.set_color("black")
    for i in range(nR):
        axe.axhline(i, color="#c8c8c8", ls="--", lw=0.35, zorder=0)
    for j in range(nG):
        axe.axvline(j, color="#c8c8c8", ls="--", lw=0.35, zorder=0)
    axe.axvline(1.5, color=GREY, ls="--", lw=0.6, zorder=0)
    SMAX = 81
    for i, ct in enumerate(ROWS):
        for j, g in enumerate(E_GENES):
            axe.scatter(j, i, s=SMAX * float(FRAC.at[gene_id[g], ct]) + 0.3, c=[float(MEAN.at[gene_id[g], ct])], cmap=CMAP_EXPR, vmin=0, vmax=emax, edgecolors="#555555", linewidths=0.25, zorder=3)
    colx = lambda j: EX0 + (j + 0.5) * (EX1 - EX0) / nG
    for j, g in enumerate(E_GENES):
        t = fig.text(fx(colx(j)), fy(BOT + 4), g, rotation=90, ha="center", va="top", fontsize=FS, fontweight="bold")
        if not N_GO_BP[gene_id[g]]:
            t.set_bbox(dict(boxstyle="square,pad=0.15", fc="none", ec=RED, lw=0.6))
    for (a_, b_), lab in [((0, 1), "TAD 1"), ((2, nG - 1), "TAD 2")]:
        xa, xb = colx(a_) - 8, colx(b_) + 8
        fig.add_artist(mpl.lines.Line2D([fx(xa), fx(xa), fx(xb), fx(xb)], [fy(417), fy(413), fy(413), fy(417)], color=PINK, lw=0.8))
        fig.text(fx((xa + xb) / 2), fy(411), lab, ha="center", va="bottom", fontsize=FS, fontweight="medium")
    fig.text(fx(EX0 + 3), fy(350), "Mean expression in group\n(log1p CP10k)", fontsize=FS, va="top", ha="left", linespacing=1.05)
    cax = fig.add_axes(box(EX0 + 10, 372, EX0 + 105, 377))
    cb = fig.colorbar(mpl.cm.ScalarMappable(cmap=CMAP_EXPR, norm=Normalize(0, emax)), cax=cax, orientation="horizontal", ticks=[0, 1, 2])
    cb.outline.set_visible(False)
    cax.tick_params(labelsize=FS, length=1.5, width=0.4, pad=1)
    fig.text(fx(272), fy(350), "Cells in group (%)", fontsize=FS, va="top", ha="left")
    lax = fig.add_axes(box(272, 362, 343, 392))
    lax.set_xlim(272, 343)
    lax.set_ylim(392, 362)
    lax.axis("off")
    for k_, fr in enumerate([0.25, 0.5, 0.75, 1.0]):
        xk = 279 + k_ * 17
        lax.scatter(xk, 371, s=SMAX * fr + 0.3, color="#999999", edgecolors="#555555", linewidths=0.25, clip_on=False)
        lax.text(xk, 382, f"{int(100 * fr)}", va="top", ha="center", fontsize=FS)
    save_fig4(fig, "fig4de_combined")
    return sd_int


with mpl.rc_context():
    mpl.rcdefaults()
    mpl.rcParams.update(FIG4_RC)
    sd_int = draw_fig4de()
display(G.round(3))
print("max. 95% CI width of the TAD importances shown:", {k: round(v, 3) for k, v in CIW.items()})
display(pd.DataFrame({g: [f"{MEAN.at[gene_id[g], ct]:.2f} ({100 * FRAC.at[gene_id[g], ct]:.0f}%)" for ct in ROWS] for g in E_GENES + ADJ}, index=[NICE4[c] for c in ROWS]))

## Figure 4f

In [ ]:
pos = positions[positions["Gene stable ID"].isin([gene_id[g] for g in KLR_CORE + ADJ])].copy()
pos["gene"] = pos["Gene stable ID"].map(name)
pos = pos.drop_duplicates("gene").set_index("gene")[["Chromosome/scaffold name", "Gene start (bp)", "Gene end (bp)"]]
pos.columns = ["chrom", "start", "end"]
chip_locus = pd.read_csv(f"{CHIP}/klr_locus_chip_fc.csv")
best = chip_locus.groupby(["factor", "accession"]).max_fold_change.max().reset_index().sort_values("max_fold_change", ascending=False).drop_duplicates("factor").set_index("factor")
dtab = dropout_table(T2, "CD8+_T_TIGIT+_CD45RO+").set_index("dropped_gene")
TOPF = ["CTCF", "RAD21"]
display(pos.sort_values("start"))
display(best)


def draw_fig4f():
    PT = 0.75
    FS, FS_CELL, FS_CB = 7.51 * PT, 6.1 * PT, 9.39 * PT
    X0F, Y0F, WF, HF = 340, 355, 336, 300
    G0, G1, PX0, PX1 = 10220000, 10580000, 381, 661
    gx = lambda b: PX0 + (np.asarray(b) - G0) / (G1 - G0) * (PX1 - PX0)
    fig = plt.figure(figsize=(WF / 96, HF / 96))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(X0F, X0F + WF)
    ax.set_ylim(Y0F + HF, Y0F)
    ax.axis("off")
    ax.plot([PX0, PX1], [379, 379], color="black", lw=0.6)
    for b in [10250000, 10350000, 10450000, 10550000]:
        ax.plot([gx(b), gx(b)], [379, 375], color="black", lw=0.6)
        ax.text(gx(b), 373, f"{b:,}", ha="center", va="bottom", fontsize=FS)
    ax.text(343, 379, "Chr 12:", ha="left", va="center", fontsize=FS, fontweight="bold")
    k0, k1 = gx(pos.loc[KLR_CORE, "start"].min()), gx(pos.loc[KLR_CORE, "end"].max())
    ax.add_patch(Rectangle((k0, 388), k1 - k0, 12, fc=NAVY, ec=PINK, lw=0.9))
    ax.text((k0 + k1) / 2, 394, "TAD 2: KLR TAD", ha="center", va="center", fontsize=FS, fontweight="bold", color="white")
    lanes = {"KLRD1": 0, "KLRK1": 0, "KLRC4": 0, "KLRC3": 0, "KLRC1": 0, "LINC02446": 0, "KLRK1-AS1": 1, "KLRC2": 1}
    LY = {0: (408, 420), 1: (446, 458)}
    LBL = {0: 429, 1: 467}
    norm = Normalize(0, 0.5)
    CW = 7.51 * 0.62
    labs = {0: [], 1: []}
    for g in KLR_CORE:
        s_, e_ = float(gx(pos.loc[g, "start"])), float(gx(pos.loc[g, "end"]))
        sh = float(dtab.loc[g, "share"]) if g in dtab.index else np.nan
        y0, y1 = LY[lanes[g]]
        ec = RED if not N_GO_BP[gene_id[g]] else GREEN
        ax.add_patch(Rectangle((s_, y0), e_ - s_, y1 - y0, fc=CMAP_DROP(norm(max(sh, 0.0))) if np.isfinite(sh) else "#eeeeee", ec=ec, lw=0.9))
        labs[lanes[g]].append(dict(g=g, mid=(s_ + e_) / 2, x=(s_ + e_) / 2, w=len(g) * CW, y1=y1))

    def spread(items, gap=5):
        items = sorted(items, key=lambda d: d["x"])
        for _ in range(300):
            moved = False
            for a_, b_ in zip(items[:-1], items[1:]):
                need = (a_["w"] + b_["w"]) / 2 + gap
                if b_["x"] - a_["x"] < need - 1e-9:
                    d = (need - (b_["x"] - a_["x"])) / 2
                    a_["x"] -= d
                    b_["x"] += d
                    moved = True
            if not moved:
                break
        return items

    for lane in (0, 1):
        for d in spread(labs[lane]):
            ax.text(d["x"], LBL[lane], d["g"], ha="center", va="top", fontsize=FS, fontweight="bold")
            ax.plot([d["mid"], d["x"]], [d["y1"], LBL[lane] - 1], color="#777777", lw=0.35)
    ax.text(580, 438, "Share of TAD gain lost\nwithout the gene in\nCD8⁺ T TIGIT⁺ CD45RO⁺", ha="left", va="top", fontsize=FS_CELL, linespacing=1.05)
    cax = fig.add_axes([(582 - X0F) / WF, 1 - (467 - Y0F) / HF, 60 / WF, 5 / HF])
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap=CMAP_DROP, norm=norm), cax=cax, orientation="horizontal", ticks=[0, 0.25, 0.5])
    cb.ax.set_xticklabels(["0%", "25%", "50%"], fontweight="bold")
    cb.outline.set_visible(False)
    cax.tick_params(labelsize=FS_CB, length=1.5, width=0.4, pad=1)
    sel = {f: chip_locus[chip_locus.accession == best.loc[f, "accession"]].sort_values("bin_start") for f in TOPF}
    ymax = float(np.ceil(max(z.max_fold_change.max() for z in sel.values()) / 20) * 20)
    for (f, z), (ty0, ty1) in zip(sel.items(), [(483, 541), (553, 611)]):
        xs = gx(z.bin_start.values + 500)
        ys = ty1 - z.max_fold_change.values / ymax * (ty1 - ty0)
        ax.fill_between(xs, ty1, ys, step="mid", color=TEAL, lw=0)
        ax.plot(xs, ys, drawstyle="steps-mid", color=NAVY, lw=0.3)
        ax.plot([PX0 - 2, PX0 - 2], [ty0, ty1], color="black", lw=0.5)
        for v in np.linspace(0, ymax, 3):
            yv = ty1 - v / ymax * (ty1 - ty0)
            ax.plot([PX0 - 4, PX0 - 2], [yv, yv], color="black", lw=0.5)
            ax.text(PX0 - 6, yv, f"{v:g}", ha="right", va="center", fontsize=FS)
        ax.text(354, (ty0 + ty1) / 2, f"{f}\nfold change\nover control", rotation=90, ha="center", va="center", fontsize=FS, fontweight="bold", linespacing=1.05)
    items = [(RED, "Gene not annotated\nwith GO terms"), (GREEN, "Gene annotated\nwith GO terms"), (PINK, "TAD 2\n(KLR TAD)")]
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    leg = []
    for ec_, txt in items:
        t = ax.text(0, 627, txt, ha="left", va="top", fontsize=FS, linespacing=1.05)
        leg.append((ec_, t, t.get_window_extent(renderer=rend).width * WF / (fig.get_figwidth() * fig.dpi)))
    BW, PAD, GAP = 17, 4, 14
    total = sum(BW + PAD + w for _, _, w in leg) + GAP * (len(leg) - 1)
    x_ = X0F + (WF - total) / 2
    for ec_, t, w in leg:
        ax.add_patch(Rectangle((x_, 627), BW, 12, fc="none", ec=ec_, lw=0.9))
        t.set_x(x_ + BW + PAD)
        x_ += BW + PAD + w + GAP
    save_fig4(fig, "fig4f_klr_locus")
    return ymax, sel


with mpl.rc_context():
    mpl.rcdefaults()
    mpl.rcParams.update(FIG4_RC)
    ymax, sel = draw_fig4f()
display(dtab.loc[[g for g in KLR_CORE if g in dtab.index]].round(3))
print("track y-axis maximum:", ymax, {f: round(z.max_fold_change.max(), 2) for f, z in sel.items()})

## Confounders

In [ ]:
groups_k562 = np.load(f"{PRIMARY}/preprocessing/resources/contact_domains/encode/ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups.npy", allow_pickle=True)
scores_k562 = np.load(f"{PRIMARY}/preprocessing/resources/contact_domains/encode/ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_score_gene_groups.npy")
hic = {}
for t in tad_ids_ordered:
    genes = t.split("_")
    for i, g in enumerate(groups_k562):
        if all(x in g for x in genes):
            hic[t] = scores_k562[i]
            break
FEATURES = {"size": pd.Series({t: len(t.split("_")) for t in tad_ids_ordered}, dtype=float), "hic_score": pd.Series(hic, dtype=float),
            "hvg_dispersion": pd.Series({t: np.nanmean([HVG_DISPERSION.get(g, np.nan) for g in t.split("_")]) for t in tad_ids_ordered})}
gain_avg = pd.Series(mean_gain_per_tad, index=tad_ids_ordered)
CONF = {}
for k, v in FEATURES.items():
    x, y = v.reindex(tad_ids_ordered).values.astype(float), gain_avg.values
    ok = ~(np.isnan(x) | np.isnan(y))
    CONF[k] = scipy.stats.spearmanr(x[ok], y[ok])
VARMEAN = scipy.stats.spearmanr(D.var(axis=1).values, D.mean(axis=1).values)
pd.DataFrame({k: [CONF[k].statistic, CONF[k].pvalue] for k in FEATURES} | {"variance vs mean": [VARMEAN.statistic, VARMEAN.pvalue]}, index=["Spearman rho", "p"]).T

In [ ]:
def save_supp(stem):
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{stem}.pdf"), bbox_inches="tight")
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{stem}.svg"), bbox_inches="tight", format="svg")
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{stem}.png"), bbox_inches="tight", dpi=500)
    plt.show()


tad_sizes = FEATURES["size"].reindex(tad_ids_ordered).values.astype(int)
set_figure_params("single", 2.2)
fig, ax = plt.subplots()
rng = np.random.default_rng(0)
for s_ in np.unique(tad_sizes):
    mask = tad_sizes == s_
    ax.scatter(np.full(mask.sum(), s_) + rng.normal(0, 0.1, mask.sum()), mean_gain_per_tad[mask], alpha=0.4, s=3, color="#70acc0")
ax.set_xlabel("Number of genes in TAD")
ax.set_ylabel("Mean TAD importance gain")
ax.set_title(f"Spearman ρ = {CONF['size'].statistic:.2f}, p = {CONF['size'].pvalue:.3e}")
plt.tight_layout()
save_supp("tad_size_vs_importance_gain")
for key, xlabel, stem in [("hic_score", "Hi-C contact domain score", "hic_score_vs_tad_importance_gain"), ("hvg_dispersion", "Mean TAD gene variability (dispersions_norm)", "hvg_score_vs_tad_importance_gain")]:
    x = FEATURES[key].reindex(tad_ids_ordered).values.astype(float)
    ok = ~np.isnan(x)
    set_figure_params("single", 2.2)
    fig, ax = plt.subplots()
    ax.scatter(x[ok], mean_gain_per_tad[ok], alpha=0.4, s=3, color="#70acc0")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Mean TAD importance gain")
    ax.set_title(f"Spearman ρ = {CONF[key].statistic:.3f}, p = {CONF[key].pvalue:.3e}")
    plt.tight_layout()
    save_supp(stem)

## Variance and mean of TAD importance gain

In [ ]:
all_data = D.values
assert VARMEAN.pvalue < 0.05
set_figure_params("single", 2.5)
plt.figure()
plt.scatter(np.var(all_data, axis=1), np.mean(all_data, axis=1), alpha=0.5, s=3, label=f"Spearman r = {VARMEAN.statistic:.2f}, p<0.05")
plt.xlabel("Variance of TAD importance across cell types")
plt.ylabel("Mean TAD importance change across cell types")
plt.legend(loc="upper center", bbox_to_anchor=(0.8, 1.1), ncol=1)
plt.tight_layout()
fig_s20, ax_s20 = plt.gcf(), plt.gca()
fig_s20.canvas.draw()
tb = fig_s20.get_tightbbox(fig_s20.canvas.get_renderer())
pad = mpl.rcParams["savefig.pad_inches"]
pts = {}
for k, t in {"TAD 1": T1, "TAD 2": T2}.items():
    vx, my = float(np.var(D.loc[t].values)), float(np.mean(D.loc[t].values))
    px, py = ax_s20.transData.transform((vx, my))
    pts[k] = dict(tad_id=t, variance=vx, mean=my, x_frac=(px / fig_s20.dpi - (tb.x0 - pad)) / (tb.width + 2 * pad), y_frac_from_top=((tb.y1 + pad) - py / fig_s20.dpi) / (tb.height + 2 * pad))
pd.DataFrame(pts).T.to_csv(os.path.join(_OUTDIR_ABS, "corr_var_example_tad_points.csv"))
save_supp("corr_var_cell_type_tad_mean")
pd.DataFrame(pts).T

## TADs with the largest and smallest gains

In [ ]:
diffs = D.T.values
examples = {"_".join(sorted(t.split("_"))) for t in [T1, T2]}
top_idx = np.argsort(mean_gain_per_tad)[::-1][:10]
top_ids = [tad_ids_ordered[i] for i in top_idx]
heatmap_data = pd.DataFrame(diffs[:, top_idx], index=phenotypes_labels, columns=[tad_to_label(t) for t in top_ids])
set_figure_params("double", 5)
fig, ax = plt.subplots()
sns.heatmap(heatmap_data, cmap=cmap_heat, center=0, vmin=-0.05, vmax=0.2, ax=ax, linewidths=0.3, linecolor="white", cbar_kws={"label": "TAD importance gain", "shrink": 0.6})
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=4.5)
for col_idx, t in enumerate(top_ids):
    if "_".join(sorted(t.split("_"))) in examples:
        ax.add_patch(plt.Rectangle((col_idx, 0), 1, len(phenotypes_labels), fill=False, edgecolor="#c12075", lw=1.5, clip_on=False))
plt.tight_layout()
save_supp("top10_tad_importance_heatmap")
named_mask = np.array([any(g in gene_name for g in t.split("_")) for t in tad_ids_ordered])
bottom_idx = np.argsort(mean_gain_per_tad * named_mask.astype(float) + (~named_mask).astype(float) * 999)[:10]
bottom_ids = [tad_ids_ordered[i] for i in bottom_idx]
heatmap_data_bottom = pd.DataFrame(diffs[:, bottom_idx], index=phenotypes_labels, columns=[tad_to_label(t) for t in bottom_ids])
set_figure_params("double", 5)
fig, ax = plt.subplots()
sns.heatmap(heatmap_data_bottom, cmap=cmap_heat, center=0, vmin=-0.05, vmax=0.05, ax=ax, linewidths=0.3, linecolor="white", cbar_kws={"label": "TAD importance gain", "shrink": 0.6})
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=4.5)
plt.tight_layout()
save_supp("bottom10_tad_importance_heatmap")
display(pd.DataFrame({"TAD": [tad_to_label(t) for t in top_ids], "mean_gain": mean_gain_per_tad[top_idx]}))
display(pd.DataFrame({"TAD": [tad_to_label(t) for t in bottom_ids], "mean_gain": mean_gain_per_tad[bottom_idx]}))